# Урок 24. Защита информации и шифрование

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 23](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-23.ipynb) · [Урок 25 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-25.ipynb)

---

Симметричное и асимметричное шифрование на понятном уровне. Хеширование и электронная подпись. Практика: реализация простых шифров.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-24", name=ФИО, klass=Класс)

## Разбираемся

### Три разные задачи

«Защита информации» — не одна задача, а три, и решаются они
разными средствами:

| Задача | Вопрос | Средство |
|---|---|---|
| **конфиденциальность** | никто посторонний не прочитает? | шифрование |
| **целостность** | данные не изменились по пути? | хеширование |
| **подлинность** | это точно тот, кто говорит? | электронная подпись |

Путать их нельзя: шифрование не защищает от подмены, а хеш
ничего не скрывает.

### Симметричное шифрование

Один ключ и на шифрование, и на расшифровку.

```
текст + ключ → шифр
шифр + ключ → текст
```

Быстро, просто, применяется для больших объёмов (диск, видеозвонок,
архив с паролем). Проблема одна и большая: **как передать ключ**.
По той же сети его отправлять нельзя — перехватят вместе с шифром.

### Асимметричное шифрование

Ключей **два**, и они связаны математически:

* **открытый** — можно публиковать где угодно, им **шифруют**;
* **закрытый** — хранится у владельца, им **расшифровывают**.

Зашифрованное открытым ключом расшифровывается только закрытым.
Значит, ключ для шифрования можно свободно раздавать — проблема
передачи исчезает.

Цена — скорость: асимметричное шифрование в тысячи раз медленнее.
Поэтому в реальности используют **обе схемы**: асимметричной
передают ключ, а дальше работают быстрой симметричной. Именно это
происходит в первые миллисекунды каждого соединения с замочком.

### Хеширование

> **Хеш** — короткая «свёртка» данных: любой объём превращается
> в строку фиксированной длины.

Свойства, из-за которых хеш полезен:

1. **необратимость** — по хешу нельзя восстановить данные;
2. **чувствительность** — изменение одного бита меняет хеш целиком;
3. **практическая неповторимость** — найти два разных текста
   с одним хешем невероятно трудно.

Отсюда применения: проверка целостности файла, хранение паролей
(в базе лежит хеш, а не пароль), и та самая проверка ответов
в этом учебнике.

### Электронная подпись

Подпись объединяет хеш и асимметричное шифрование:

1. автор считает хеш документа;
2. **шифрует хеш своим закрытым ключом** — это и есть подпись;
3. получатель расшифровывает подпись открытым ключом автора
   и сравнивает с хешем документа, который посчитал сам.

Совпало — значит, документ не изменён **и** подписан владельцем
закрытого ключа. Обратите внимание: здесь ключи работают
в обратную сторону — шифрует закрытый, проверяет открытый.

### Классические шифры

Они давно не защищают ничего, но на них видно механику.

**Цезарь.** Сдвиг каждой буквы на фиксированное число.
Ключей всего 25 — перебираются мгновенно.

**Виженер.** Сдвиг переменный: задаётся ключевым словом, которое
повторяется вдоль текста. Триста лет считался невзламываемым.

**XOR.** Каждый символ складывается с ключом побитовым
исключающим «или». Замечательное свойство: та же операция
расшифровывает — потому что `(a ^ k) ^ k == a`.

### Почему они ломаются

Любой шифр простой замены сохраняет **частоты**. Самая частая
буква шифра — самая частая буква языка. Метод придумал арабский
математик аль-Кинди в IX веке, и он до сих пор первое, что
пробуют на незнакомом шифре.

## Смотрим, как это работает

### Пример 1. Шифр Цезаря

In [ ]:
АЛФАВИТ = "абвгдежзийклмнопрстуфхцчшщъыьэюя"


def цезарь(текст, сдвиг):
    результат = ""
    for символ in текст.lower():
        если_есть = АЛФАВИТ.find(символ)
        if если_есть == -1:
            результат += символ
        else:
            результат += АЛФАВИТ[(если_есть + сдвиг) % len(АЛФАВИТ)]
    return результат


сообщение = "атака на рассвете"
шифр = цезарь(сообщение, 3)

print("Открытый текст:", сообщение)
print("Зашифровано:   ", шифр)
print("Расшифровано:  ", цезарь(шифр, -3))

### Пример 2. Взлом Цезаря перебором

In [ ]:
print("Перебираем все сдвиги:")
for сдвиг in range(1, 8):
    print(f"  {сдвиг:>2}: {цезарь(шифр, -сдвиг)}")

Тридцать два варианта — и один из них осмысленный. Компьютеру
на это нужны микросекунды, а раньше требовался человек, знающий
язык.

### Пример 3. Шифр Виженера

In [ ]:
def виженер(текст, ключ, расшифровать=False):
    результат = ""
    позиция = 0
    for символ in текст.lower():
        индекс = АЛФАВИТ.find(символ)
        if индекс == -1:
            результат += символ
            continue
        сдвиг = АЛФАВИТ.find(ключ[позиция % len(ключ)])
        if расшифровать:
            сдвиг = -сдвиг
        результат += АЛФАВИТ[(индекс + сдвиг) % len(АЛФАВИТ)]
        позиция += 1
    return результат


шифр_в = виженер("атака на рассвете", "ключ")

print("Виженер:      ", шифр_в)
print("Расшифровано: ", виженер(шифр_в, "ключ", расшифровать=True))
print()
print("Одна и та же буква «а» превратилась в разные:")
print("  видно, что простым перебором сдвига это не вскрыть.")

### Пример 4. XOR: шифрование и расшифровка одной операцией

In [ ]:
def xor_шифр(данные, ключ):
    результат = bytearray()
    for i in range(len(данные)):
        результат.append(данные[i] ^ ключ[i % len(ключ)])
    return bytes(результат)


текст = "секретное сообщение".encode("utf-8")
ключ = "пароль".encode("utf-8")

зашифровано = xor_шифр(текст, ключ)
расшифровано = xor_шифр(зашифровано, ключ)

print("Байты шифра:", зашифровано[:12].hex(" "))
print("Расшифровано:", расшифровано.decode("utf-8"))
print("Совпало:", текст == расшифровано)

### Пример 5. Хеш и его чувствительность

In [ ]:
import hashlib


def хеш(текст):
    return hashlib.sha256(текст.encode("utf-8")).hexdigest()


for текст in ["Привет", "привет", "Привет!", "Привет "]:
    print(f"{текст!r:<10} → {хеш(текст)[:32]}…")

Четыре почти одинаковых строки — четыре совершенно разных хеша.
Ни малейшего сходства: изменение одного символа меняет всё.

### Пример 6. Проверка целостности

In [ ]:
файл = "Важный документ. Перевести 100 рублей."
подделка = "Важный документ. Перевести 900 рублей."

эталон = хеш(файл)

print("Хеш оригинала:", эталон[:24], "…")
print("Хеш подделки: ", хеш(подделка)[:24], "…")
print()
print("Документ подлинный:", хеш(файл) == эталон)
print("Подделка пройдёт проверку:", хеш(подделка) == эталон)

### Пример 7. Пароли хранят хешем

In [ ]:
база = {"ivanov": хеш("qwerty123"), "petrova": хеш("оченьсложный")}


def проверить_вход(логин, пароль):
    return база.get(логин) == хеш(пароль)


print("Верный пароль:   ", проверить_вход("ivanov", "qwerty123"))
print("Неверный пароль: ", проверить_вход("ivanov", "qwerty124"))
print()
print("В базе хранится:", база["ivanov"][:24], "…")
print("Даже украв базу, паролей не узнать — хеш необратим.")

Честная оговорка: простые пароли всё равно подбирают по словарю
заранее посчитанных хешей. Поэтому к паролю добавляют случайную
«соль» и хешируют медленными функциями — но идея та же.

## Пробуем сами

### Задача 1. Сколько ключей

Сколько ключей используется в асимметричном шифровании?

In [ ]:
#@title 🧩 Задача 1. Число ключей { display-mode: "form" }
#@markdown Впиши число
ключей = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", ключей, "d4735e3a265e16ee",
         hint="Один можно публиковать, второй — нет.")

### Задача 2. Каким ключом шифруют

В асимметричной схеме каким ключом **шифруют** сообщение
для получателя?

In [ ]:
#@title 🧩 Задача 2. Ключ шифрования { display-mode: "form" }
#@markdown Выбери ключ
ключ_шифрования = "выбери ответ" #@param ["выбери ответ", "открытый ключ", "закрытый ключ"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", ключ_шифрования, "5e58622a55cf42c2",
         hint="Тот, который можно свободно раздавать.")

### Задача 3. Что обеспечивает хеширование

Какая из трёх задач защиты решается хешированием?

In [ ]:
#@title 🧩 Задача 3. Задача хеширования { display-mode: "form" }
#@markdown Выбери задачу
задача = "выбери ответ" #@param ["выбери ответ", "конфиденциальность", "целостность", "доступность"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("3", задача, "4d64335daef7b306",
         hint="Хеш ничего не скрывает — он показывает, что данные не менялись.")

### Задача 4. Шифр Цезаря

Функция сдвигает буквы русского алфавита на заданное число.
Небуквенные символы оставляйте как есть, текст приводите
к строчным буквам.

Алфавит лежит в переменной `АЛФАВИТ`.

In [ ]:
def шифр_цезаря(текст, сдвиг):
    return ...

In [ ]:
si.check("4", шифр_цезаря, [
    (("абв", 1), "бвг"),
    (("яблоко", 1), "авмплп"),
    (("а б", 2), "в г"),
    (("абв", 0), "абв"),
])

### Задача 5. XOR туда и обратно

Функция применяет XOR к последовательности байтов с ключом.
Она же должна и расшифровывать: проверка применяет её дважды.

In [ ]:
def xor(данные, ключ):
    return ...

In [ ]:
def дважды(текст, ключ):
    данные = текст.encode("utf-8")
    к = ключ.encode("utf-8")
    return xor(xor(данные, к), к).decode("utf-8")


si.check("5", дважды, [
    (("секрет", "ключ"), "секрет"),
    (("hello", "k"), "hello"),
    (("", "k"), ""),
])

### Задача 6. Проверка целостности

Функция получает текст и эталонный хеш (полный SHA-256
в шестнадцатеричном виде) и возвращает `True`, если текст
не изменён.

In [ ]:
def целостность(текст, эталон):
    return ...

In [ ]:
эталонный = hashlib.sha256("Важный документ".encode("utf-8")).hexdigest()

si.check("6", целостность, [
    (("Важный документ", эталонный), True),
    (("Важный документ!", эталонный), False),
    (("", эталонный), False),
])

## Домашнее задание

### Домашнее задание 1. Взлом частотным анализом

Зашифруйте шифром Цезаря текст хотя бы на 500 символов (возьмите
любой отрывок). Затем, **не пользуясь ключом**, взломайте его:

1. посчитайте частоты букв в шифре;
2. предположите, что самая частая буква — это «о»;
3. найдите нужный сдвиг и проверьте.

Объясните, почему этот способ не работает на шифре Виженера.
Покажите работу учителю.

### Домашнее задание 2. Виженер

Функция шифрует текст ключевым словом: сдвиг каждой буквы задаётся
соответствующей буквой ключа, ключ повторяется по кругу.

Небуквенные символы оставляются как есть и **не сдвигают**
позицию в ключе.

In [ ]:
def шифр_виженера(текст, ключ):
    return ...

In [ ]:
si.check("дз2", шифр_виженера, [
    (("абв", "а"), "абв"),
    (("абв", "б"), "бвг"),
    (("аа аа", "аб"), "аб аб"),
    (("", "ключ"), ""),
])

### Домашнее задание 3. Схема для больших объёмов

Какая схема шифрования быстрее и потому применяется
для больших объёмов данных?

In [ ]:
#@title 🏠 Домашнее 3. Быстрая схема { display-mode: "form" }
#@markdown Выбери схему
схема = "выбери ответ" #@param ["выбери ответ", "симметричное", "асимметричное"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", схема, "36c1061e26a067a0",
         hint="Та, где ключ один.")

---

### Любопытно

Асимметричное шифрование придумали дважды. В 1976 году его
опубликовали американцы Диффи и Хеллман, и это считается началом
современной криптографии.

Но за шесть лет до них ту же идею нашли в британском Центре
правительственной связи — и засекретили. Автор, Джеймс Эллис,
умер в 1997 году, за несколько недель до того, как разработку
рассекретили. Он так и не получил признания при жизни.

Мораль отчасти про науку, отчасти про открытость: идея, которую
опубликовали, изменила мир, а идея, которую спрятали, тридцать лет
лежала в сейфе.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 23](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-23.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 25 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-25.ipynb)